In [ ]:
from tensorflow.keras.applications import EfficientNetV2S

def build_efficientnet_model():
    # Load EfficientNetV2-S with ImageNet weights
    base_model = EfficientNetV2S(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3),
        include_preprocessing=True # Built-in scaling! No need for rescale=1/255
    )

    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.4)(x)
    # Output layer for 3 classes: Normal, Pneumonia, Covid
    output = Dense(3, activation="softmax")(x)

    model = Model(inputs=base_model.input, outputs=output)
    return model, base_model

# Build and compile
model, base_model = build_efficientnet_model()

# Stage 2 (Fine-tuning) is critical for EfficientNet:
# Unfreeze the last 50 layers for specific medical feature adaptation
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)